# Chuyển BiGRU Alphabet sang TFLite FP32

Mở notebook trong Google Colab, chọn **Runtime → Run all**, upload checkpoint khi được hỏi, rồi gửi lại file ZIP kết quả.

Kiến trúc lấy nguyên từ VOYA-Collector; model hidden=64, 2 GRU layers, dropout=0.3 được kiểm chứng bằng weights. Nhãn lấy từ checkpoint. Không quantize/retrain. Chuỗi test có cấu trúc landmark kiểm tra độ khớp số học; accuracy thực cần thêm `test_data.npz`.

Dependencies chỉ cài trong Colab. Không cần bật GPU. [LiteRT Torch](https://github.com/google-ai-edge/litert-torch), [torch/torchao compatibility](https://github.com/pytorch/ao/issues/2919).

In [ ]:
# Chỉ cài trong máy Linux của Colab. CPU đủ dùng; không cần GPU.
# Dùng virtualenv riêng để tránh xung đột torch/numpy có sẵn của Colab.
import os, sys, subprocess
from pathlib import Path
assert sys.platform == "linux", "Cell này dành cho Google Colab/Linux."
assert (3, 10) <= sys.version_info[:2] <= (3, 13), "Chọn runtime Python 3.10–3.13."
os.chdir("/content")
VENV = Path("/content/alphabet_converter_env")
if not (VENV / "bin/python").exists():
    subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=False)
PYTHON = str(VENV / "bin/python")
pip_ready = (VENV / "bin/python").exists() and subprocess.run(
    [PYTHON, "-m", "pip", "--version"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode == 0
if not pip_ready:
    # Một số image Colab thiếu ensurepip/python-venv; không cần apt hoặc restart.
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv==20.35.3"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(VENV)], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "torch==2.11.0", "torchao==0.17.0",
                "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
subprocess.run([PYTHON, "-m", "pip", "install", "litert-torch==0.9.4", "litert-converter==0.4.0",
                "ai-edge-litert==2.2.0", "numpy==2.1.3", "torch==2.11.0", "torchao==0.17.0"], check=True)
subprocess.run([PYTHON, "-c", "import torch, litert_torch; print('torch:', torch.__version__); print('litert_torch:', litert_torch.__version__)"], check=True)
print("Đã chuẩn bị xong môi trường conversion.")


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
os.chdir("/content")
CHECKPOINT = Path("bigru_attention_alphabet_20260818_114440.pt")
if not CHECKPOINT.exists():
    print("Chọn file bigru_attention_alphabet_20260818_114440.pt từ máy của bạn.")
    files.upload()
assert CHECKPOINT.exists(), f"Thiếu {CHECKPOINT.name}"
assert hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() == "0f6626b2857827645c7139e4870d02cfd905e5fcf9c3b4d879d266415f0237eb", "Checkpoint không phải file đã kiểm tra trong dự án."
print("Checkpoint đúng SHA-256.")
# Không bắt buộc: nếu có bộ test thực, upload test_data.npz bằng panel Files.
# Định dạng: x float32 [N,60,126] đã normalize đúng lúc train;
# y int [N] là class index từ checkpoint (0..29), optional sample_ids [N].
TEST_DATA = Path("/content/test_data.npz")


## Python converter độc lập
Cell dưới ghi file Python; có thể thu gọn cell sau khi chạy.

In [ ]:
%%writefile /content/convert_alphabet_colab.py
# Generated by scripts/alphabet/build_colab.py. Standalone, original BiGRU architecture.
# Run on Colab/Linux: python convert_alphabet_colab.py convert --output-dir output_run_1
from __future__ import annotations

# Original source: VOYA-Collector/processed/train_utils/models/base.py
"""Base class cho tất cả sign language models"""

from abc import ABC, abstractmethod
from typing import Any, Dict, Optional

import torch
import torch.nn as nn


def initialize_kaiming(module: nn.Module) -> None:
    """
    Initialize all Conv/Linear/RNN layers with Kaiming Normal (He et al., 2015).

    Kaiming initialization is standard for ReLU networks and ensures:
    - Consistent weight distribution across architectures
    - Proper variance scaling based on network depth
    - Fair comparison between different model types

    Reference: He et al. "Delving Deep into Rectifiers" - ICCV 2015

    Args:
        module: PyTorch module to initialize
    """
    for m in module.modules():
        if isinstance(m, (nn.Conv1d, nn.Conv2d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.LSTM, nn.GRU)):
            # For recurrent layers, initialize weights (not biases)
            for name, param in m.named_parameters():
                if 'weight_ih' in name or 'weight_hh' in name:
                    nn.init.kaiming_normal_(param, nonlinearity='relu')
                elif 'bias' in name:
                    nn.init.zeros_(param)
        elif isinstance(m, nn.BatchNorm1d):
            # BatchNorm: weights to 1, biases to 0
            if m.weight is not None:
                nn.init.ones_(m.weight)
            if m.bias is not None:
                nn.init.zeros_(m.bias)


class SignLanguageModel(ABC, nn.Module):
    """
    Abstract base class cho tất cả sign language models.

    Tất cả models phải implement:
    - forward(x) để inference
    - from_config() để tạo model từ config dict
    - get_model_name() để lấy tên model
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        name: Optional[str] = None,
        **kwargs,
    ):
        """
        Args:
            input_dim: Số features đầu vào (thường 126 cho sign language)
            output_dim: Số classes đầu ra
            name: Tên model (nếu None dùng class name)
        """
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self._model_name = name or self.__class__.__name__

    @abstractmethod
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass

        Args:
            x: Input tensor shape (batch, seq_len, input_dim)

        Returns:
            Output logits shape (batch, output_dim)
        """
        pass

    @classmethod
    @abstractmethod
    def from_config(
        cls,
        input_dim: int,
        output_dim: int,
        config: Optional[Dict[str, Any]] = None,
    ):
        """
        Tạo model từ config dict.

        Args:
            input_dim: Input dimension
            output_dim: Output dimension
            config: Dict chứa model-specific hyperparameters
                   (ví dụ: dropout, channels, levels, kernel_size, etc.)

        Returns:
            Model instance
        """
        pass

    def get_model_name(self) -> str:
        """Lấy tên model để log"""
        return self._model_name

    def get_config(self) -> Dict[str, Any]:
        """
        Trả về config của model.
        Override trong subclass nếu cần lưu hyperparameters.
        """
        return {
            "model": self.get_model_name(),
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
        }

    def count_parameters(self) -> int:
        """Đếm tổng số parameters"""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def __repr__(self) -> str:
        """Nice representation"""
        param_count = self.count_parameters()
        return f"{self.get_model_name()}(input_dim={self.input_dim}, output_dim={self.output_dim}, params={param_count:,})"


# Original source: VOYA-Collector/processed/train_utils/models/bigru_attention.py
"""Bidirectional GRU with Attention for sign language recognition"""

from typing import Any, Dict, Optional

import torch
import torch.nn as nn



class AttentionLayer(nn.Module):
    """
    Scaled Dot-Product Attention Mechanism (Vaswani et al., 2017).

    Computes: Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
    where d_k is the dimension of keys (hidden_size).

    Reference: "Attention Is All You Need" - NIPS 2017
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.query = nn.Linear(hidden_size, hidden_size)
        self.key = nn.Linear(hidden_size, hidden_size)
        self.value = nn.Linear(hidden_size, hidden_size)
        self.scale = hidden_size ** 0.5

        # Kaiming initialization for attention projections
        for module in [self.query, self.key, self.value]:
            nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Compute scaled dot-product attention.

        Args:
            x: [B, T, hidden_size] input sequence

        Returns:
            [B, T, hidden_size] attention-weighted output
        """
        Q = self.query(x)  # [B, T, hidden_size]
        K = self.key(x)    # [B, T, hidden_size]
        V = self.value(x)  # [B, T, hidden_size]

        # Scaled dot-product attention: softmax(Q @ K^T / sqrt(d_k)) @ V
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # [B, T, T]
        attn_weights = torch.softmax(scores, dim=-1)                # [B, T, T]
        context = torch.matmul(attn_weights, V)                     # [B, T, hidden_size]

        return context


class BiGRUAttentionModel(SignLanguageModel):
    """
    Bidirectional GRU with Scaled Dot-Product Attention (Vaswani et al., 2017).

    Architecture:
    - Bidirectional GRU layers for bidirectional sequence processing
    - Scaled dot-product attention mechanism for temporal focus
    - Global average pooling over time for fixed-size representation
    - Linear classifier head

    Why this architecture for sign language:
    - BiGRU processes both past and future context (crucial for gesture understanding)
    - Attention weights temporal importance (some frames matter more for classification)
    - Combined: captures long-range dependencies + temporal focus
    - Fewer parameters than multi-head attention, good for small datasets

    Computational complexity:
    - BiGRU: O(T * hidden_size^2) for T time steps
    - Attention: O(T^2 * hidden_size) for T^2 attention matrix
    - Total: Efficient for sequence lengths T < 100 (typical for sign language)
    """

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        hidden_size: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
        **kwargs,
    ):
        """
        Args:
            input_dim: Input dimension (typically 126 for sign language)
            output_dim: Number of output classes
            hidden_size: GRU hidden units
            num_layers: Number of GRU layers (1-4)
            dropout: Dropout rate between GRU layers
        """
        super().__init__(input_dim, output_dim, name="BiGRU + Attention")
        self.hidden_size = hidden_size
        self.num_layers = max(1, min(4, num_layers))
        self.dropout_rate = dropout

        # Bidirectional GRU layers
        # Output: [B, T, hidden_size * 2]
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_size,
            num_layers=self.num_layers,
            dropout=dropout if self.num_layers > 1 else 0.0,
            bidirectional=True,
            batch_first=True,
        )

        # Attention mechanism
        gru_output_dim = hidden_size * 2  # bidirectional
        self.attention = AttentionLayer(gru_output_dim)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(gru_output_dim, gru_output_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(gru_output_dim // 2, output_dim),
        )

        # Initialize weights with Kaiming Normal (He et al., 2015)
        initialize_kaiming(self)

    def encode(self, x_btd: torch.Tensor) -> torch.Tensor:
        """
        Encode sequence to fixed-size representation with attention.

        Args:
            x_btd: Input tensor [B, T, D]
                   B = batch size
                   T = sequence length
                   D = input dimension (126)

        Returns:
            Pooled representation [B, gru_output_dim]
        """
        if x_btd.ndim != 3:
            raise RuntimeError(f"Expected 3D tensor [B,T,D], got {x_btd.shape}")

        # GRU forward pass
        # gru_out: [B, T, hidden_size * 2]
        gru_out, _ = self.gru(x_btd)

        # Apply attention
        # attn_out: [B, T, hidden_size * 2]
        attn_out = self.attention(gru_out)

        # Global average pooling over time dimension
        # [B, T, hidden_size * 2] -> [B, hidden_size * 2]
        return attn_out.mean(dim=1)

    def forward(self, x_btd: torch.Tensor) -> torch.Tensor:
        """
        Forward pass.

        Args:
            x_btd: Input tensor [B, T, D]

        Returns:
            Class logits [B, output_dim]
        """
        return self.classifier(self.encode(x_btd))

    @classmethod
    def from_config(
        cls,
        input_dim: int,
        output_dim: int,
        config: Optional[Dict[str, Any]] = None,
    ) -> "BiGRUAttentionModel":
        """
        Create BiGRU+Attention model from config dict.

        Args:
            input_dim: Input dimension (126)
            output_dim: Number of classes
            config: Dict with keys:
                - hidden_size (int): GRU hidden units. Default: 64
                - num_layers (int): Number of GRU layers (1-4). Default: 2
                - dropout (float): Dropout rate. Default: 0.3

        Returns:
            BiGRUAttentionModel instance
        """
        if config is None:
            config = {}

        return cls(
            input_dim=input_dim,
            output_dim=output_dim,
            hidden_size=config.get("hidden_size", 64),
            num_layers=config.get("num_layers", 2),
            dropout=config.get("dropout", 0.3),
        )

    def get_config(self) -> Dict[str, Any]:
        """Get model configuration for logging/saving"""
        return {
            "model": "BiGRU + Attention",
            "input_dim": self.input_dim,
            "output_dim": self.output_dim,
            "hidden_size": self.hidden_size,
            "num_layers": self.num_layers,
            "dropout": self.dropout_rate,
        }


# Original source: VOYA-Collector/processed/shared/normalization.py
import numpy as np

def normalize_single_hand(hand: np.ndarray) -> np.ndarray:
    """
    Normalize ONE hand independently.

    hand shape: (21,3)
    """

    h = hand.astype(np.float32).copy()

    # empty hand
    if not np.any(h):
        return h

    # wrist landmark
    wrist = h[0, :2].copy()

    # translate
    h[:, :2] = h[:, :2] - wrist

    # compute scale
    valid = np.linalg.norm(h[:, :2], axis=1) > 1e-6

    if valid.any():

        pts = h[valid, :2]

        span_x = pts[:,0].max() - pts[:,0].min()
        span_y = pts[:,1].max() - pts[:,1].min()

        scale = max(span_x, span_y)

        if scale > 1e-6:
            h[:, :2] = h[:, :2] / scale

    return h

def normalize_hands_vector_126(vec: np.ndarray) -> np.ndarray:

    if vec is None:
        return vec

    v = np.asarray(vec, dtype=np.float32)

    if v.size != 126:
        return v

    try:
        arr = v.reshape(2, 21, 3).astype(np.float32)
    except Exception:
        return v

    # preserve semantic hand identity
    left = arr[0]
    right = arr[1]

    # normalize independently
    left = normalize_single_hand(left)
    right = normalize_single_hand(right)

    out = np.concatenate([
        left.reshape(-1),
        right.reshape(-1)
    ]).astype(np.float32)

    return out

EMBEDDED_CONTRACT_JSON = '{\n  "checkpointSha256": "0f6626b2857827645c7139e4870d02cfd905e5fcf9c3b4d879d266415f0237eb",\n  "modelConfig": {"hidden_size": 64, "num_layers": 2, "dropout": 0.3},\n  "configEvidence": {\n    "reason": "Legacy train_tcn.build_checkpoint writes TCN fields even for BiGRU. Do not interpret levels=3 as GRU layers.",\n    "hidden_size": "gru.weight_hh_l0 shape [192,64]: 3 gates, H=64",\n    "num_layers": "Only l0/l1 and both reverse directions exist in model_state_dict",\n    "dropout": "checkpoint.model_config.dropout and training_config.dropout are both 0.3",\n    "source": "VOYA-Collector/processed/train_utils/models/bigru_attention.py"\n  },\n  "sampleFps": 30,\n  "sampleFpsEvidence": "VOYA-Collector/frontend/src/config/capture.ts: SAMPLE_FPS=30, TARGET_FRAMES=60; historical capture timestamps not available",\n  "swapHandedness": true,\n  "mirrorInput": false,\n  "orientationEvidence": [\n    "VOYA-Collector/frontend/src/utils/realtimeFlatten.ts: toDetections swaps raw MP labels once",\n    "VOYA-Collector/frontend/src/config/handTracking.ts: MIRROR_SERVING_PAYLOAD=false",\n    "VOYA-Collector/backend/realtime_service/app/contracts.py: swapped_mp_handedness_slots"\n  ],\n  "normalizationVersion": "alphabet_hands126_v1",\n  "normalization": "Swap raw MP hand blocks once; XY subtract each wrist, divide by max span of non-wrist valid XY points; keep Z unchanged. No rounding. Missing landmarks remain zero; missing wrist invalidates that hand.",\n  "missingFramePolicy": "Timestamp bins at 30 FPS; zero-fill missing bins and empty detections, no carry-forward; restart after a gap longer than one window. Never predict when the current detection is empty.",\n  "limitation": "Training sources contain differing capture/video missing-frame and mirror policies. This pins the serving contract; empirical accuracy/orientation still needs the checkpoint\'s validation/test data."\n}\n'
MODEL_SOURCE_SHA256 = {'VOYA-Collector/processed/train_utils/models/base.py': '1cd3150297c54d41caa84d12545fd67dd7b4312f7a38ca64110a3d8143df1484', 'VOYA-Collector/processed/train_utils/models/bigru_attention.py': '79c8d4f82dbb2e51cf05126c518fc0ac2a4853dfdea59dc3310c5372a8cc5da1', 'VOYA-Collector/processed/shared/normalization.py': '2c416100a9aca58be3cf5f0bf31f6559dd69ba089eaf12355dee8311828028bc'}
"""Strict checkpoint audit, FP32 LiteRT conversion, multi-input parity and staging.

All project inputs/outputs must be inside this workspace. The converter is Linux-only;
`verify` also runs on Windows against an existing TFLite without claiming conversion.
"""

import argparse
import hashlib
import importlib.metadata
import json
import platform
import re
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parent

STEM = "bigru_attention_alphabet_20260818_114440"
ASSETS = ROOT / "android/app/src/main/assets"
ATOL, RTOL = 1e-4, 1e-4


def local(value: str | Path) -> Path:
    p = (ROOT / value).resolve()
    if not p.is_relative_to(ROOT):
        raise ValueError(f"Path outside workspace is forbidden: {p}")
    return p


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def write_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def versions() -> dict:
    result = {"python": sys.version, "platform": platform.platform()}
    for package in ("torch", "numpy", "litert-torch", "litert-converter", "torchao", "ai-edge-litert", "tensorflow"):
        try:
            result[package] = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            result[package] = None
    return result


def load_checkpoint(path: Path, contract: dict):
    if sha(path) != contract["checkpointSha256"]:
        raise ValueError("Checkpoint hash differs from reviewed deployment_contract.json")
    c = torch.load(path, map_location="cpu", weights_only=True)
    required = {"seq_len", "feature_dim", "num_classes", "model_config", "model_state_dict",
                "idx_to_label", "label_to_idx", "preprocess_contract", "normalization_version"}
    if required - c.keys():
        raise ValueError(f"Missing checkpoint metadata: {required - c.keys()}")
    assert (c["seq_len"], c["feature_dim"], c["num_classes"]) == (60, 126, 30)
    assert c["model_type"] == "BiGRU + Attention"
    assert c["normalization_version"] == "hands126_v1"
    assert c["preprocess_contract"]["expects_strict_shape"] == [60, 126]
    assert c["preprocess_contract"]["coordinate_order"] == "xyz"
    cfg = contract["modelConfig"]
    for key in ("hidden_size", "num_layers", "dropout"):
        if key in c["model_config"] and c["model_config"][key] != cfg[key]:
            raise ValueError(f"Checkpoint config disagrees with manifest: {key}")
    state = c["model_state_dict"]
    h = cfg["hidden_size"]
    assert tuple(state["gru.weight_hh_l0"].shape) == (3 * h, h)
    layers = {int(m.group(1)) for key in state if (m := re.fullmatch(r"gru.weight_ih_l(\d+)", key))}
    assert layers == set(range(cfg["num_layers"]))
    model = BiGRUAttentionModel(input_dim=c["feature_dim"], output_dim=c["num_classes"], **cfg).cpu().eval()
    model.load_state_dict(state, strict=True)
    rich = {int(k): v for k, v in c["idx_to_label"].items()}
    assert set(rich) == set(range(c["num_classes"]))
    labels = {str(i): rich[i]["label_original"] for i in range(c["num_classes"])}
    assert len(set(labels.values())) == c["num_classes"]
    assert len(c["label_to_idx"]) == c["num_classes"]
    for i, item in rich.items():
        assert c["label_to_idx"][item["label_key"]] == i
    return c, model, labels


def preprocess(raw: np.ndarray, contract: dict) -> np.ndarray:
    raw = np.asarray(raw, dtype=np.float32)
    if raw.shape != (126,) or not np.isfinite(raw).all():
        raise ValueError("Expected one finite raw MediaPipe frame [126]")
    hands = raw.reshape(2, 21, 3).copy()
    if contract["swapHandedness"]:
        hands = hands[::-1].copy()
    out = np.zeros_like(hands)
    for j, hand in enumerate(hands):
        present = np.any(hand != 0, axis=1)
        if not present[0]:
            continue
        if contract["mirrorInput"]:
            hand[present, 0] = 1 - hand[present, 0]
        xy = hand[:, :2] - hand[0, :2]
        valid = present & (np.sum(xy * xy, axis=1) > np.float32(1e-12))
        scale = np.float32(1)
        if valid.any():
            candidate = np.ptp(xy[valid], axis=0).max()
            if candidate > 1e-6:
                scale = candidate
        out[j, present, :2] = xy[present] / scale
        out[j, present, 2] = hand[present, 2]
    if not np.isfinite(out).all():
        raise ValueError("Alphabet normalization overflow")
    return out.reshape(126)


def fixtures(contract: dict):
    # Articulated hands, changing finger flexion, orientation, translation and depth.
    # These test numerical parity, not recognition accuracy of synthetic signs.
    rng = np.random.default_rng(42)
    names, raws = [], []
    kinds = ("raw_mp_left", "raw_mp_right", "both", "empty", "lost_frames", "negative_z", "off_origin_wrist", "missing_landmark")
    for kind in kinds:
        for variant in range(8):
            clip = np.zeros((60, 2, 21, 3), np.float32)
            angles = rng.uniform(-0.8, 0.8, 2)
            for t in range(60):
                phase = 2 * np.pi * t / 60
                for side in range(2):
                    if kind == "empty" or (kind == "raw_mp_left" and side == 1) or (kind == "raw_mp_right" and side == 0):
                        continue
                    if kind == "lost_frames" and (t % 11 in (0, 1) or (side == 0 and 20 <= t < 35)):
                        continue
                    wrist = np.array([0.28 + side * 0.40 + .03 * np.sin(phase), .68 + .025 * np.cos(phase), 0], np.float32)
                    if kind == "off_origin_wrist":
                        wrist[:2] += [-.12, -.23]
                    hand = np.zeros((21, 3), np.float32)
                    hand[0] = wrist
                    theta = angles[side] + .12 * np.sin(phase)
                    rotation = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
                    for finger in range(5):
                        flex = .3 + .65 * (.5 + .5 * np.sin(phase + finger + variant))
                        for joint in range(4):
                            idx = 1 + finger * 4 + joint
                            xy = np.array([(finger - 2) * .029 + (joint * .007 if finger == 0 else 0), -.045 - (joint + 1) * .034 * flex])
                            xy[0] *= 1 if side == 0 else -1
                            hand[idx, :2] = wrist[:2] + rotation @ xy
                            hand[idx, 2] = -.008 * (joint + 1) * (1 + flex)
                    if kind == "missing_landmark":
                        hand[6] = 0
                        if variant == 7:
                            hand[0] = 0
                    clip[t, side] = hand
            names.append(f"{kind}_{variant}")
            raws.append(clip.reshape(60, 126))
    raw = np.stack(raws)
    x = np.stack([[preprocess(f, contract) for f in clip] for clip in raw])
    # Verify original training function on its valid full-hand domain.
    for clip in raw[:24]:
        for frame in clip:
            ordered = frame.reshape(2, 63)[::-1].reshape(126) if contract["swapHandedness"] else frame
            if not contract["mirrorInput"]:
                np.testing.assert_allclose(preprocess(frame, contract), normalize_hands_vector_126(ordered), atol=2e-6, rtol=2e-6)
    return names, raw, x


def read_validation(path: Path):
    """Explicit evaluation bundle: x float32 [N,60,126], y integer indices, optional sample_ids.

    x is already normalized by the training loader; never normalize it again.
    """
    with np.load(path, allow_pickle=False) as d:
        x, y = np.asarray(d["x"], np.float32), np.asarray(d["y"])
        if x.ndim != 3 or x.shape[1:] != (60, 126) or not np.isfinite(x).all():
            raise ValueError("Evaluation x must be finite [N,60,126]")
        if y.shape != (len(x),) or y.dtype.kind not in "iu" or np.any((y < 0) | (y >= 30)) or not len(x):
            raise ValueError("Evaluation y must be checkpoint class indices [N], 0..29")
        names = d["sample_ids"].astype(str).tolist() if "sample_ids" in d else [f"test_{i}" for i in range(len(x))]
        if len(names) != len(x):
            raise ValueError("sample_ids length mismatch")
    return names, x, y


def interpreter(path: Path):
    from ai_edge_litert.interpreter import Interpreter
    i = Interpreter(model_path=str(path), num_threads=4)
    i.allocate_tensors()
    sigs = i.get_signature_list()
    assert len(sigs) == 1, sigs
    key, sig = next(iter(sigs.items()))
    assert len(sig["inputs"]) == len(sig["outputs"]) == 1
    inp, out = i.get_input_details(), i.get_output_details()
    assert len(inp) == len(out) == 1
    for tensor, shape in ((inp[0], [1, 60, 126]), (out[0], [1, 30])):
        assert tensor["shape"].tolist() == shape
        assert tensor["dtype"] == np.float32
        assert tensor["quantization"] == (0.0, 0)
        assert len(tensor["quantization_parameters"]["scales"]) == 0
    def describe(t):
        return {"name": t["name"], "shape": t["shape"].tolist(), "dtype": str(t["dtype"]), "quantization": list(t["quantization"])}
    metadata = {"signatureKey": key, "frameInputName": sig["inputs"][0], "outputName": sig["outputs"][0],
                "signatures": sigs, "input": describe(inp[0]), "output": describe(out[0])}
    runner = i.get_signature_runner(key)
    def predict(x):
        return runner(**{metadata["frameInputName"]: x})[metadata["outputName"]]
    return i, metadata, predict


def compare(expected, actual, names):
    a, b = np.asarray(expected, np.float64), np.asarray(actual, np.float64)
    assert a.shape == b.shape and np.isfinite(a).all() and np.isfinite(b).all()
    error = np.abs(a - b)
    relative = error / np.maximum(np.abs(a), 1e-6)
    changed = np.flatnonzero(a.argmax(-1) != b.argmax(-1))
    return {"sampleCount": len(a), "argmaxAgreement": float(1 - len(changed) / len(a)),
            "maxAbsoluteError": float(error.max()), "meanAbsoluteError": float(error.mean()),
            "maxRelativeError": float(relative.max()), "meanRelativeError": float(relative.mean()),
            "relativeDenominatorFloor": 1e-6, "atol": ATOL, "rtol": RTOL,
            "allclose": bool(np.allclose(a, b, atol=ATOL, rtol=RTOL)),
            "changedArgmax": [{"sample": names[j], "expected": int(a[j].argmax()), "actual": int(b[j].argmax())} for j in changed]}


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("command", choices=("inspect", "verify", "convert"))
    p.add_argument("--checkpoint", default=f"{STEM}.pt")
    p.add_argument("--contract", help="Optional override of the embedded, hash-pinned contract")
    p.add_argument("--tflite", default=f"android/app/src/main/assets/{STEM}_fp32.tflite")
    p.add_argument("--output-dir", required=True, help="New directory; refuses to overwrite a previous report")
    p.add_argument("--test-data", help="Workspace NPZ with normalized x [N,60,126], y [N] and optional sample_ids")
    args = p.parse_args()
    out = local(args.output_dir)
    if out.exists():
        raise FileExistsError(f"Use a NEW output directory: {out}")
    out.mkdir(parents=True)
    torch.manual_seed(42)
    torch.set_num_threads(4)
    torch.use_deterministic_algorithms(True)
    contract = json.loads(local(args.contract).read_text(encoding="utf-8")) if args.contract else json.loads(EMBEDDED_CONTRACT_JSON)
    c, model, labels = load_checkpoint(local(args.checkpoint), contract)
    write_json(out / "checkpoint.json", {k: v for k, v in c.items() if k != "model_state_dict"})
    write_json(out / "labels.json", labels)
    write_json(out / f"{STEM}_verified_labels.json", labels)
    print("Checkpoint loaded with strict=True; BiGRU hidden=64, layers=2; 30 labels verified.", flush=True)
    report = {"versions": versions(), "checkpointSha256": sha(local(args.checkpoint)), "strictLoad": True,
              "modelSourceSha256": MODEL_SOURCE_SHA256, "resolvedModelConfig": model.get_config(), "contract": contract, "historicalCheckpointMetrics": c.get("metrics"),
              "accuracy": None, "androidParity": None, "afterConvertParity": None, "newConversionExecuted": False}
    write_json(out / "report.json", report)
    if args.command == "inspect":
        write_json(out / "report.json", report)
        return
    names, raw, x = fixtures(contract)
    print(f"Prepared {len(x)} structured landmark sequences.", flush=True)
    golden_count = len(x)
    if args.test_data:
        test_names, test_x, y = read_validation(local(args.test_data))
        names += test_names
        x = np.concatenate([x, test_x])
        report["testDataSha256"] = sha(local(args.test_data))
    with torch.inference_mode():
        expected = np.concatenate([model(torch.from_numpy(clip[None])).numpy() for clip in x])
    sample = (torch.from_numpy(x[:1]),)
    # A strict export check is separate from LiteRT conversion.
    ep = torch.export.export(model, sample, strict=True)
    print("torch.export.export passed; comparing exported program.", flush=True)
    with torch.inference_mode():
        exported = np.concatenate([ep.module()(torch.from_numpy(clip[None])).numpy() for clip in x])
    report["torchExportParity"] = compare(expected, exported, names)
    write_json(out / "report.json", report)
    path = local(args.tflite)
    if args.command == "convert":
        try:
            import litert_torch
        except ImportError as e:
            report["conversionBlocker"] = "litert_torch is unavailable; official converter requires Linux."
            write_json(out / "report.json", report)
            raise RuntimeError(report["conversionBlocker"]) from e
        with torch.no_grad():
            print("Starting litert_torch.convert on CPU in FP32...", flush=True)
            edge = litert_torch.convert(model.cpu().eval(), sample)
        print("Conversion finished; checking in-memory outputs before serialization.", flush=True)
        converted = np.concatenate([np.asarray(edge(clip[None])) for clip in x])
        report["afterConvertParity"] = compare(expected, converted, names)
        write_json(out / "report.json", report)
        path = out / f"{STEM}_verified_fp32.tflite"
        edge.export(str(path))
        print(f"Serialized {path.name}; reloading in LiteRT for parity.", flush=True)
        report["newConversionExecuted"] = True
    _, metadata, predict = interpreter(path)
    actual = np.concatenate([predict(clip[None]) for clip in x])
    report["serializedTfliteParity"] = compare(expected, actual, names)
    if args.command == "convert":
        report["convertVsReloadParity"] = compare(converted, actual, names)
    report["tfliteSha256"] = sha(path)
    report["tfliteMetadata"] = metadata
    existing_labels_path = ASSETS / f"{STEM}_display_labels.json"
    report["existingLabelsMatchCheckpoint"] = (
        json.loads(existing_labels_path.read_text(encoding="utf-8")) == labels
        if existing_labels_path.exists() else None
    )
    if args.test_data:
        report["accuracy"] = {"n": len(y), "pytorch": float((expected[golden_count:].argmax(-1) == y).mean()),
                              "tflite": float((actual[golden_count:].argmax(-1) == y).mean())}
    parity_checks = [report["torchExportParity"], report["serializedTfliteParity"]]
    if report["afterConvertParity"]:
        parity_checks.append(report["afterConvertParity"])
        parity_checks.append(report["convertVsReloadParity"])
    report["pythonParityPassed"] = all(r["allclose"] and r["argmaxAgreement"] == 1 for r in parity_checks)
    write_json(out / "report.json", report)
    np.savez_compressed(out / "fixtures.npz", raw=raw, x=x, pytorch_logits=expected, tflite_logits=actual, sample_ids=np.array(names))
    raw.astype("<f4").tofile(out / "raw.f32")
    x.astype("<f4").tofile(out / "input.f32")
    write_json(out / "golden.json", {"modelSha256": sha(path), "checkpointSha256": report["checkpointSha256"],
               "sampleNames": names, "rawSampleCount": golden_count, "shape": list(x.shape),
               "labels": labels, "pytorchLogits": expected.tolist(), "tfliteLogits": actual.tolist(),
               "rawSha256": sha(out / "raw.f32"), "inputSha256": sha(out / "input.f32"), "atol": ATOL, "rtol": RTOL})
    # Generate a reviewable registry entry from the actual binary metadata. No activation here.
    write_json(out / "registry-entry.json", {
        "displayName": "BiGRU Attention Alphabet FP32 (verified)", "modelFile": f"{STEM}_verified_fp32.tflite",
        "labelsFile": f"{STEM}_verified_labels.json", "sequenceLength": 60, "featureDimension": 126, "classCount": 30,
        "signatureKey": metadata["signatureKey"], "frameInputName": metadata["frameInputName"], "lengthInputName": "",
        "outputName": metadata["outputName"], "inputTensorName": metadata["input"]["name"], "outputTensorName": metadata["output"]["name"],
        "applySoftmax": True, "mirrorInput": contract["mirrorInput"], "swapHandedness": contract["swapHandedness"],
        "normalizationVersion": contract["normalizationVersion"], "sampleFps": contract["sampleFps"],
        "numThreads": 4, "modelSha256": sha(path), "labelsSha256": sha(out / "labels.json"), "outputType": "logits"
    })
    print(json.dumps({k: report[k] for k in ("pythonParityPassed", "serializedTfliteParity", "accuracy", "newConversionExecuted")}, indent=2))
    if not report["pythonParityPassed"]:
        raise RuntimeError("PARITY FAILED: do not activate this model")


if __name__ == "__main__":
    main()


In [ ]:
import datetime, json, subprocess, shutil
from pathlib import Path
from google.colab import files
RUN = Path("/content") / ("alphabet_export_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
LOG = Path(str(RUN) + ".log")
cmd = [PYTHON, "/content/convert_alphabet_colab.py", "convert", "--checkpoint", str(CHECKPOINT.resolve()), "--output-dir", str(RUN)]
if TEST_DATA.exists():
    cmd += ["--test-data", str(TEST_DATA)]
print("Đang convert FP32 và kiểm tra 64 golden sequences. Có thể mất vài phút.")
# Lưu toàn bộ log, kể cả lỗi import/export/conversion/parity.
with LOG.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    exit_code = process.wait()
RUN.mkdir(exist_ok=True)
shutil.copyfile(LOG, RUN / "conversion.log")
with (RUN / "requirements-frozen.txt").open("w") as f:
    subprocess.run([PYTHON, "-m", "pip", "freeze"], stdout=f, check=True)
report_path = RUN / "report.json"
report = json.loads(report_path.read_text()) if report_path.exists() else {}
passed = exit_code == 0 and report.get("pythonParityPassed") is True and report.get("newConversionExecuted") is True
status = "PASS" if passed else "FAILED"
if passed:
    for stage in ["afterConvertParity", "serializedTfliteParity", "convertVsReloadParity"]:
        print(stage, json.dumps(report[stage], indent=2))
    print("PASS: TFLite FP32 đã vượt qua kiểm tra PyTorch → convert → serialize/load lại.")
else:
    print("FAILED: chưa sử dụng model này trong app. Gửi ZIP chứa log lại để kiểm tra.")
archive = shutil.make_archive(str(RUN) + "_" + status, "zip", root_dir=RUN)
print("Tải ZIP này về và gửi lại trong cuộc trò chuyện:", Path(archive).name)
files.download(archive)
